> ## SOLUTION / ANSWER KEY &mdash; Lab 6.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-model-and-prompt.ipynb`](../lab-03-model-and-prompt.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.3 &mdash; The Model Interface: Prompts & .invoke()

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Build a PromptTemplate with an {input} slot and format it
- Call the real ChatOllama model with .invoke() and read .content
- See that the SAME interface would work for ChatGroq (model-agnostic)

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; LangChain's core building blocks](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-03"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Every LangChain chat model shares **one interface**: `model.invoke(prompt)` returns a message whose
text is in **`.content`**. That is why swapping **`ChatOllama`** for **`ChatGroq`** is a one-line change.
A **`PromptTemplate`** (from `langchain_core.prompts`) fills slots like `{input}`. Here you build the
prompt and call the **real** model.

In [ ]:
from langchain_core.prompts import PromptTemplate

demo_prompt = PromptTemplate.from_template("Say hi to {who}.")
print("formatted:", demo_prompt.format(who="Ada"))
print("the setup cell already built `llm` =", llm.model)

## Build it
Build a **template** with an `{input}` slot, a `build_prompt` that fills it, and `ask` that calls the
**real** `llm` and returns `.content`.

In [ ]:
from langchain_core.prompts import PromptTemplate

def build_template():
    return PromptTemplate.from_template("Answer in one concise sentence.\nQuestion: {input}")

def build_prompt(question):
    return build_template().format(input=question)

def ask(question):
    """Call the real model and return its text."""
    reply = llm.invoke(build_prompt(question))
    return reply.content

try:
    print("input variables:", build_template().input_variables)
    print("prompt sent:", build_prompt("what is an AI agent?"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Call the real model through your `ask()` and read the answer it returns.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        print("Q: In one sentence, what is an AI agent?")
        print("A:", ask("In one sentence, what is an AI agent?"))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The reply came back through **`.invoke(prompt).content`** &mdash; the one interface.
- Swap `llm` for `ChatGroq(model="llama-3.3-70b-versatile")` (with a `GROQ_API_KEY`) and `ask()` is unchanged. That's what "model-agnostic" buys you.

## Your turn (open task &mdash; no grader)
Change the template &mdash; make it answer as bullet points, or in French, or as a five-year-old &mdash; and
re-run `ask()`. Then (optional) build a second model `groq = ChatGroq(model="llama-3.3-70b-versatile")`
and call `groq.invoke(build_prompt(...)).content`. **What good looks like:** the answer's *style* changes
with the template, and the *same* `.invoke().content` call works across models.

---
### Key takeaway
One interface -- .invoke(prompt).content -- over every provider. The PromptTemplate shapes the request; swapping the model is one line.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>